# Part 1 · Foundations

**Qubits · quantum states · measurement · expectation values · entanglement.**

Quantum computers exist because some computational problems scale badly on classical machines. Simulating molecules and materials (the basis of drug discovery and battery design) and searching large combinatorial spaces (scheduling, routing, portfolio selection) both grow so fast with problem size that classical hardware runs out of memory or time. A few tasks are impossible classically rather than merely expensive: generating randomness that is provably unpredictable, or communicating in a way that exposes any eavesdropper. This notebook builds the vocabulary that those applications rest on.

We treat quantum states as **mathematical objects** that you construct and probe directly, using only numpy and QiliSDK's state primitive. There are no circuits and no simulator yet; those arrive in Part 2. By the end you will be able to:

- describe a qubit's state as a vector of complex **amplitudes**;
- predict **measurement** outcomes with the **Born rule**, and compute **expectation values** of observables;
- create and quantify **superposition** and **entanglement**.

No prior quantum-computing knowledge is assumed, only Python and a little linear algebra: vectors, matrices, and complex numbers.

### How QiliSDK is organized

Every part of the tutorial sits somewhere on the same three-layer map, so it is worth fixing in mind now:

1. **Primitives** are the building blocks: `QTensor` (states and operators), `Gate`/`Circuit` (digital), `Hamiltonian`/`Schedule` (analog), and `Readout` (what to measure).
2. **Workflows** are *functionals* that wrap a primitive into something runnable: `DigitalPropagation`, `AnalogEvolution`, `VariationalProgram`.
3. **Execution** is handled by *backends* that run a workflow on a simulator or real hardware: CPU (`QiliSim`), GPU (`CudaBackend`), or a quantum processing unit, i.e. a real QPU (`SpeQtrum`).

Part 1 works entirely in the Primitives layer, with the most basic primitive of all: **`QTensor`**, the quantum state itself.

In [ ]:
# ▶ Run me first. No-op if QiliSDK is already installed; installs it on a fresh env (e.g. Google Colab).
try:
    import qilisdk
except ImportError:
    import sys
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "qilisdk[openqasm,qir]==0.2.1", "matplotlib", "numpy"], check=True)
    import qilisdk  # Colab: if this still fails, Runtime > Restart session, then rerun
print("QiliSDK", qilisdk.__version__)

In [ ]:
%matplotlib inline
import numpy as np

from qilisdk.core import QTensor, ket, bra, ghz, tensor_prod, expect_val

## 1.1 · Qubits and quantum states

A **classical bit** is either $0$ or $1$. A **qubit** state is, concretely, something you already know how to store: a length-2 numpy array of complex numbers, `np.array([alpha, beta], dtype=complex)`, normalized so that `np.sum(np.abs(v)**2) == 1`. That is the entire data structure. A **ket** is that same array shaped as a column vector, shape `(2, 1)`; a **bra** is its conjugate transpose, `ket.conj().T`, a row vector of shape `(1, 2)`. Two quick glosses before they appear everywhere: the *complex conjugate* of $a+bi$ is $a-bi$ (flip the sign of the imaginary part), and the *dagger* symbol $\dagger$ means conjugate transpose, `.conj().T` in numpy.

Physicists write all this in *Dirac notation* (also called bra-ket notation), which is just typed syntax around those column and row vectors:

$$|\psi\rangle = \alpha\,|0\rangle + \beta\,|1\rangle,\qquad \alpha,\beta\in\mathbb{C},\qquad |\alpha|^2 + |\beta|^2 = 1.$$

Here $|\psi\rangle$ (read "ket psi") is the state, and $|0\rangle$ and $|1\rangle$ are the two *basis states*, represented as column vectors

$$|0\rangle = \begin{pmatrix}1\\0\end{pmatrix},\qquad |1\rangle = \begin{pmatrix}0\\1\end{pmatrix}.$$

The complex numbers $\alpha,\beta$ are called **amplitudes**. They are *not* probabilities, but the **Born rule** (made concrete in §1.2) turns them into probabilities. The constraint $|\alpha|^2+|\beta|^2=1$ is just the statement that those probabilities sum to one.

In QiliSDK, states and operators are both represented by a single object: the **`QTensor`** (quantum tensor). It is the `Core` primitive everything else is built on, and you can explore it with no simulator at all. `ket(0)` gives $|0\rangle$, `ket(1)` gives $|1\rangle$, and `bra(0)` gives the bra $\langle 0| = |0\rangle^\dagger$ (that conjugate-transpose row vector).

In [ ]:
psi = ket(0)                        # the |0> state
print("state vector:", psi.dense().ravel())  # amplitudes [1, 0]
print("nqubits:", psi.nqubits, " is_ket:", psi.is_ket())

### Combining qubits: the tensor product

Two qubits do **not** live in two separate 2-D spaces glued together: they live in their *tensor product* (written $\otimes$), a single $2 \times 2 = 4$-dimensional space with basis $|00\rangle, |01\rangle, |10\rangle, |11\rangle$. Concretely, the tensor product multiplies the two qubits' amplitudes pairwise: if qubit A is $[a_0, a_1]$ and qubit B is $[b_0, b_1]$, the joint state is the length-4 vector $[a_0 b_0,\; a_0 b_1,\; a_1 b_0,\; a_1 b_1]$, one amplitude for each combination of the two qubits' values. In numpy this is exactly `np.kron(A, B)`. Every additional qubit multiplies the number of amplitudes by two, so an $n$-qubit state is a vector of $2^n$ complex numbers.

QiliSDK uses **big-endian** ordering: in `ket(a, b)` the *first* argument is the most-significant qubit. So `ket(0, 1)` is $|01\rangle$, which is basis index $1$ (the bitstring `01` read as the binary number 1) in that length-4 vector:

In [ ]:
q = ket(0, 1)                      # the 2-qubit state |01>
print("dimension:", q.shape, "(= 2**2 x 1)")
print("|01> is basis index:", np.argmax(np.abs(q.dense())))   # 1  (big-endian)
print("a bra is the conjugate-transpose:", bra(0).dense())    # row vector [1, 0]

### The $2^n$ memory wall: why quantum computers should exist

In 1982 Richard Feynman made what is often called the founding argument for quantum computing. Simulating a quantum system on a classical computer means storing all $2^n$ of its amplitudes, and that memory cost grows exponentially with the number of particles, yet nature evolves the same system effortlessly inside every molecule. His proposal was to turn the difficulty around: to simulate a quantum system, build hardware that is itself quantum, whose physical state holds those $2^n$ amplitudes directly. At its core this is a memory-complexity argument, and the next cell makes the numbers concrete (each amplitude is a `complex128`, so 16 bytes):

In [ ]:
print("Storing one quantum state in classical RAM (complex128 = 16 bytes per amplitude):")
for n in [1, 2, 4, 8, 12, 20]:
    amps = ket(*([0] * n)).dense()   # ket(0, 0, ..., 0) = the n-qubit |00...0> state
    print(f"  {n:>2} qubits -> {amps.size:>9,} amplitudes = {amps.nbytes:>12,} bytes")

print("Extrapolating with plain Python (16 * 2**n bytes):")
print(f"  33 qubits -> {16 * 2**33 / 1e9:,.0f} GB   (a very large workstation)")
print(f"  50 qubits -> {16 * 2**50 / 1e15:,.0f} PB  (about twice the memory of the Frontier supercomputer)")
print(f"  280 qubits -> {16 * 2**280:.1e} bytes (more than the ~1e80 atoms in the observable universe)")

### The catch: you cannot read the amplitudes out

Storing $2^n$ amplitudes might sound like free exponential memory, but it is not, because you cannot read those amplitudes out. When you *measure* $n$ qubits (§1.2) you get back exactly $n$ classical bits, one $0/1$ per qubit, and the $2^n$-amplitude state is destroyed in the process. The exponential state is physically real, but it is not addressable memory you can inspect entry by entry. Every useful quantum algorithm must therefore arrange **interference** so that the amplitudes of wrong answers cancel while the amplitudes of right answers reinforce, leaving the few bits you *do* read out worth reading. Parts 3 and 4 show two different ways of doing this.

### Superposition

The defining quantum feature is **superposition**: a qubit can be in a combination of $|0\rangle$ *and* $|1\rangle$ at once. The most important examples are the *plus* and *minus* states, both equal superpositions:

$$|+\rangle = \tfrac{1}{\sqrt{2}}\big(|0\rangle + |1\rangle\big),\qquad |-\rangle = \tfrac{1}{\sqrt{2}}\big(|0\rangle - |1\rangle\big).$$

Both have amplitudes of magnitude $1/\sqrt 2$, so both give outcomes $0$ and $1$ with probability $\tfrac12$ when measured in the **computational basis** (the standard $|0\rangle/|1\rangle$ basis; it is also called the $Z$ basis, after the observable introduced in §1.3). Yet they are **different states**: they differ in the *sign* of their second amplitude, and more generally in its complex *phase*, which is physical. That relative phase is what makes **interference** possible: when amplitudes are later recombined, terms with opposite signs can cancel. We cannot show that cancellation yet, because it requires *operating* on the state, which is Part 2's subject; here we only need to establish that $|+\rangle$ and $|-\rangle$ are genuinely distinct states even though they give identical measurement statistics. (An overall *global* phase, such as multiplying the whole vector by $-1$, is **not** observable and does not count as a difference between states.)

The code below builds $|+\rangle$ two equivalent ways and $|-\rangle$ once, and confirms that $|+\rangle$ and $|-\rangle$ share the same amplitude magnitudes but differ in the sign of the second amplitude.

In [ ]:
plus = QTensor.uniform(1)                                   # built-in equal superposition |+>
print("|+> built-in          :", np.round(plus.dense().ravel(), 3))            # [0.707 0.707]

# The same state, built by hand by adding the basis kets and normalizing:
plus_by_hand = (ket(0) + ket(1)).normalized()
print("|+> = (|0>+|1>)/sqrt2 :", np.round(plus_by_hand.dense().ravel(), 3))     # [0.707 0.707]

# |-> differs from |+> ONLY in the sign of the second amplitude:
minus = (ket(0) - ket(1)).normalized()
print("|-> = (|0>-|1>)/sqrt2 :", np.round(minus.dense().ravel(), 3))            # [ 0.707 -0.707]

### Visualizing a qubit: the Bloch sphere

Any single-qubit *pure* state can be written, up to global phase, as

$$|\psi\rangle = \cos\tfrac{\theta}{2}\,|0\rangle + e^{i\varphi}\sin\tfrac{\theta}{2}\,|1\rangle,$$

so it maps to a point on a unit sphere, the **Bloch sphere**, with polar angle $\theta$ and azimuth $\varphi$. Matching this to our amplitudes gives $\alpha = \cos(\theta/2)$ and $\beta = e^{i\varphi}\sin(\theta/2)$. The factor $e^{i\varphi}$ is the phase: a point on the unit circle in the complex plane (Euler's formula, $e^{i\varphi} = \cos\varphi + i\sin\varphi$). The angle is *halved* for a concrete geometric reason: $|0\rangle$ and $|1\rangle$ are orthogonal as vectors (90° apart in the state space), but the picture places them at opposite poles of the sphere (180° apart), and the factor of two between $\theta$ and $\theta/2$ reconciles the two conventions. With that in place, $|0\rangle$ sits at the north pole ($\theta = 0$), $|1\rangle$ at the south pole ($\theta = \pi$), and $|+\rangle$ / $|-\rangle$ on the equator, at opposite ends of the $\pm X$ axis. `QTensor.draw()` renders any single-qubit state as its point on this sphere:

In [ ]:
QTensor.uniform(1).draw()   # |+> points along the +X axis of the Bloch sphere

### Your four reference states

A handful of single-qubit states recur throughout this tutorial and across quantum computing generally, so it helps to introduce them together. Two you already have, the **computational-basis** states $|0\rangle$ and $|1\rangle$; the other two are their equal-superposition counterparts $|+\rangle$ and $|-\rangle$. They define two natural measurement axes for a qubit:

- the **$Z$ basis** (the *computational basis*) $\{|0\rangle, |1\rangle\}$: the north and south **poles** of the Bloch sphere;
- the **$X$ basis** $\{|+\rangle, |-\rangle\}$: two opposite points on the **equator**.

The table below collects all four. Read each row as a rule: fixing the amplitudes fixes the measurement probabilities through the Born rule.

| state | amplitudes $[\alpha, \beta]$ | $P(0)$ | $P(1)$ | Bloch location |
|---|---|---|---|---|
| $\lvert 0\rangle$ | $[1,\ 0]$ | $1$ | $0$ | north pole ($+Z$) |
| $\lvert 1\rangle$ | $[0,\ 1]$ | $0$ | $1$ | south pole ($-Z$) |
| $\lvert +\rangle$ | $\tfrac{1}{\sqrt2}[1,\ 1]$ | $\tfrac12$ | $\tfrac12$ | $+X$ (equator) |
| $\lvert -\rangle$ | $\tfrac{1}{\sqrt2}[1,\ -1]$ | $\tfrac12$ | $\tfrac12$ | $-X$ (equator) |

Note the last two rows. $|+\rangle$ and $|-\rangle$ have **identical** measurement probabilities, yet they are genuinely different states. The only thing separating them is the *sign* of the second amplitude, a **relative phase**: invisible to a computational-basis measurement, but plainly visible as opposite points on the sphere. Section 1.3 introduces the tool ($\langle X\rangle$) that distinguishes them. The code below builds all four states and draws each on its own Bloch sphere.

In [ ]:
from qilisdk.utils.visualization import QTensorStyle

states = {
    "|0>": ket(0),
    "|1>": ket(1),
    "|+>": QTensor.uniform(1),                 # = (|0> + |1>)/sqrt(2)
    "|->": (ket(0) - ket(1)).normalized(),     # = (|0> - |1>)/sqrt(2)
}

for name, s in states.items():
    amps = np.round(s.dense().ravel(), 3)
    p0, p1 = [round(p, 3) for p in s.probabilities()]
    print(f"{name}:  amplitudes {amps}   P(0) = {p0}   P(1) = {p1}")

# Each state as a point on the Bloch sphere: |0>/|1> at the poles, |+>/|-> on the equator.
for name, s in states.items():
    s.draw(style=QTensorStyle(title=name))

## 1.2 · Measurement and the Born rule

A qubit holds continuous amplitudes, but when you **measure** it in the computational basis you get a single classical bit, $0$ or $1$. The **Born rule** gives the probabilities as the squared magnitudes of the amplitudes:

$$P(0) = |\alpha|^2,\qquad P(1) = |\beta|^2.$$

Measurement is also **destructive**: it *collapses* the superposition onto the basis state you observed, so measuring the same (now-collapsed) qubit again returns the same value. To gather statistics you must **prepare the state afresh and measure again** each time; that repetition is exactly what the `nshots` parameter of a real device controls (Part 2).

This already has a direct application. Measuring $|+\rangle$ repeatedly is a **provably fair coin**: the 50/50 odds come from physics, not from an algorithm that could be predicted or reverse-engineered. This is the working principle of commercial quantum random number generators (QRNGs); Part 2 builds one as its dice roller and returns there to why this physical guarantee beats a classical pseudo-random generator.

`QTensor.probabilities()` returns the Born-rule distribution directly. One caveat about the sampling line below: Part 1 has no simulator yet, so we sample *classically* from the quantum probabilities with `np.random.choice`. Real quantum measurement arrives in Part 2.

In [ ]:
plus = QTensor.uniform(1)
print("fair coin   P(0), P(1):", [round(p, 4) for p in plus.probabilities()])    # [0.5, 0.5]

theta = np.pi / 6
biased = np.cos(theta) * ket(0) + np.sin(theta) * ket(1)   # scalar * is fine for scaling
print("biased coin P(0), P(1):", [round(p, 4) for p in biased.probabilities()])  # [0.75, 0.25]

# No simulator yet: mimic 10 fresh prepare-and-measure rounds by sampling classically.
draws = np.random.choice([0, 1], size=10, p=plus.probabilities())
print("10 simulated measurements of |+>:", draws)

### 🧩 Exercise 1.1: build a loaded quantum coin

A fair quantum coin comes from equal amplitudes. This exercise builds a *biased* one by choosing the amplitudes deliberately: a single-qubit state that outputs $1$ with probability **exactly 0.75**. The bias is set at the amplitude level, so the odds are fixed by physics rather than by code that someone could tamper with.

Use the one-parameter family $|\psi(\theta)\rangle = \cos\theta\,|0\rangle + \sin\theta\,|1\rangle$, which by the Born rule gives $P(1) = \sin^2\theta$.

1. Choose `theta` so that $\sin^2\theta = 0.75$ (solve on paper, or let numpy invert it for you).
2. Build the state yourself from `ket(0)` and `ket(1)`, scaled by the right amplitudes.
3. Verify with `probabilities()` that it reads `[0.25, 0.75]`.
4. Draw 20 sample bits with `np.random.choice`.

This is the Born rule run backwards: instead of reading probabilities off a given state, you pick the probabilities you want and solve for the amplitudes that produce them. Choosing amplitudes to hit a target distribution is exactly what QRNG hardware does in silicon.

In [ ]:
# TODO: build the loaded quantum coin.
# 1) Pick theta so that np.sin(theta)**2 == 0.75.
# 2) Build the state: a cos(theta) amplitude on ket(0) plus a sin(theta) amplitude on ket(1).
# 3) Print its .probabilities() and check you get [0.25, 0.75].
# 4) Sample 20 bits: np.random.choice([0, 1], size=20, p=<your probabilities>).


## 1.3 · Expectation values

Often we do not want a full histogram, only an **average**. An **observable** is a *Hermitian* operator $O$; Hermitian means $O$ equals its own conjugate transpose (`O == O.conj().T` in numpy), which guarantees that the averages it produces are real numbers. The **eigenvalues** of $O$ are the possible measurement outcomes, and an **eigenstate** is a state whose outcome is certain: measure an eigenstate and you always get its eigenvalue. The **expectation value**

$$\langle O\rangle = \langle\psi|\,O\,|\psi\rangle$$

reads as **bra @ operator @ ket**: a row vector times a matrix times a column vector, which collapses to a single number. It is the mean outcome you would obtain by averaging many measurements of $O$ on identically prepared copies of the state.

The most common observable is **Pauli-$Z$**, with eigenvalues $+1$ (eigenstate $|0\rangle$) and $-1$ (eigenstate $|1\rangle$). So

$$\langle Z\rangle = P(0)\cdot(+1) + P(1)\cdot(-1) = P(0) - P(1) \in [-1, 1]$$

measures how "$0$-like" versus "$1$-like" a state is: it reads $+1$ on $|0\rangle$, $-1$ on $|1\rangle$, and $0$ on an equal superposition. **Pauli-$X$** measures the same quantity in the $|+\rangle / |-\rangle$ basis.

Expectation values run through the second half of the tutorial: the average energy of a molecule is one, and the optimization algorithms of Parts 3 and 4 both work by finding the state that *minimizes* such a value.

Use `expect_val(operator, state)`, and note that the **operator comes first**. The code below also computes the same number by hand, spelling out the bra–operator–ket product so the definition is explicit:

In [ ]:
Z_op = QTensor(np.array([[1, 0], [0, -1]]))     # Pauli-Z observable

for name, state in [("|0>", ket(0)), ("|1>", ket(1)), ("|+>", QTensor.uniform(1))]:
    print(f"<Z> on {name}: {round(expect_val(Z_op, state).real, 4)}")   # +1, -1, 0

# The sandwich <psi|Z|psi> written out: bra @ operator @ ket = a 1x1 matrix.
state = QTensor.uniform(1)
sandwich = state.adjoint() @ Z_op @ state       # .adjoint() = dagger = conjugate transpose
print("by hand    :", round(sandwich.dense()[0, 0].real, 4))
print("expect_val :", round(expect_val(Z_op, state).real, 4))

# And <Z> is exactly P(0) - P(1):
p = state.probabilities()
print("P(0) - P(1):", round(p[0] - p[1], 4))    # 0.0 for |+>

### 🧩 Exercise 1.2: probe the $|-\rangle$ state

$|+\rangle$ and $|-\rangle$ produce *identical* 50/50 statistics when measured in the computational basis, which raises the question of whether they differ at all. They do, and $\langle X\rangle$ is what shows it. Build $|-\rangle = \tfrac{1}{\sqrt2}(|0\rangle - |1\rangle)$ and verify that $\langle Z\rangle = 0$ (equal $0/1$ probabilities, just like $|+\rangle$) **but** $\langle X\rangle = -1$. The state $|-\rangle$ is the $-1$ eigenstate of $X$, whereas $|+\rangle$ has $\langle X\rangle = +1$, and that difference is what distinguishes them.

Hints: subtract two kets and call `.normalized()` to build the state; `expect_val(operator, state)` returns a complex number, so take `.real` and round it.

The relative phase that a $Z$ measurement cannot see is exactly the quantity that interference-based algorithms and quantum key distribution depend on. A measurement in the right basis ($X$ here) turns that hidden phase into an observable difference.

In [ ]:
Z_op = QTensor(np.array([[1, 0], [0, -1]]))
X_op = QTensor(np.array([[0, 1], [1, 0]]))

# TODO: build |-> = (|0> - |1>)/sqrt(2), then print <Z> and <X> for it.


## 1.4 · Entanglement

Some multi-qubit states **cannot be written as a product** of single-qubit states. These are *entangled*. The canonical example is the **Bell state**

$$|\Phi^+\rangle = \tfrac{1}{\sqrt 2}\big(|00\rangle + |11\rangle\big).$$

No pair of single-qubit states $|a\rangle = [a_0, a_1]$ and $|b\rangle = [b_0, b_1]$ satisfies $|\Phi^+\rangle = |a\rangle \otimes |b\rangle$. A product state has amplitudes $[a_0 b_0,\ a_0 b_1,\ a_1 b_0,\ a_1 b_1]$; matching the Bell state would require $a_0 b_0 = a_1 b_1 = \tfrac{1}{\sqrt2}$ (so all four of $a_0, a_1, b_0, b_1$ are non-zero) *and* $a_0 b_1 = a_1 b_0 = 0$ (so at least one of them is zero). Those two demands contradict each other, so no factorization exists.

The consequence shows up in the measurements. Measure the Bell state and only `00` and `11` ever appear: two parties holding one qubit each obtain a **random bit they always agree on**. Compare the unentangled state $|+\rangle \otimes |+\rangle$: there all four outcomes appear at 25%, so the two parties agree only half the time, no better than independent coin flips.

Entanglement therefore produces **correlated randomness that exists only between its holders**. Nobody chose the shared bit, it did not exist before the measurements, and (as the next demo computes) an outsider who examines one qubit alone sees pure noise. Perfectly correlated for the two insiders, featureless for anyone outside: that is exactly the property a cryptographic key needs. This resource underlies entanglement-based quantum key distribution, and the Bell-test experiments that confirmed these correlations earned the 2022 Nobel Prize in Physics. One caveat: measuring your half tells you what the other side *will* read, but on its own it transmits no information, so entanglement does not permit faster-than-light signaling.

In [ ]:
bell = ghz(2)                                 # for 2 qubits, ghz() builds the Bell state
print("Bell amplitudes     :", np.round(bell.dense().ravel(), 3))              # [0.707 0 0 0.707]
print("Bell P(00,01,10,11) :", [round(p, 4) for p in bell.probabilities()])    # [0.5, 0.0, 0.0, 0.5]

plus = QTensor.uniform(1)
prod = tensor_prod([plus, plus])              # unentangled: each qubit is its own |+>
print("|+>|+> P(00..11)    :", [round(p, 4) for p in prod.probabilities()])    # [0.25, 0.25, 0.25, 0.25]

# The same trick scales: a 3-qubit GHZ state gives three parties one shared random bit.
print("GHZ(3) P(000..111)  :", [round(p, 4) for p in ghz(3).probabilities()])  # only 000 and 111

### One level deeper: the reduced state (for the curious)

The claim "an outsider sees pure noise" can be computed exactly, using two tools that return later in the tutorial.

First, the **density matrix** $\rho = |\psi\rangle\langle\psi|$ (ket times bra, a matrix): a representation that can also express a probability-weighted *mixture* of states, like a dict mapping states to probabilities, where a ket can only express one definite vector. Part 5 gives it a full treatment; today it is just the input format for the second tool.

Second, the **partial trace** answers "what does qubit 0 look like on its own, ignoring its partner?" The convention to memorize: `partial_trace({0})` **keeps** qubit 0 and discards (traces out) all the others. (The partial trace itself returns in Part 6's quantum-reservoir capstone.)

For the Bell state the reduced state of qubit 0 comes out $\rho_A = \tfrac12 I$: a 50/50 mixture of $|0\rangle$ and $|1\rangle$ with no phase information whatsoever, a perfectly fair coin. That is the outsider's view, computed. The **von Neumann entropy**

$$S(\rho) = -\sum_i \lambda_i \ln\lambda_i$$

(where $\lambda_i$ are the eigenvalues of $\rho$) quantifies the randomness. QiliSDK measures it in **nats** (natural logarithm), so maximal one-qubit randomness is $\ln 2 \approx 0.6931$ nats, i.e. one full bit. A product state scores exactly $0.0$: no entanglement, no shared randomness. And the pattern generalizes: in a 3-qubit GHZ state, keep any single qubit and trace out the others, and that one-qubit reduced state is again $\tfrac12 I$.

In [ ]:
rho_A = bell.as_density_matrix().partial_trace({0})    # KEEPS qubit 0, traces out qubit 1
print("reduced state of qubit 0:\n", np.round(rho_A.dense(), 3))            # I/2 -> a fair coin
print("Bell entanglement entropy:", round(rho_A.entropy_von_neumann(), 4))  # 0.6931 = ln 2 (maximal)

prod00 = tensor_prod([ket(0), ket(0)])                 # |00> is a product (unentangled) state
print("product-state entropy    :",
      round(prod00.as_density_matrix().partial_trace({0}).entropy_von_neumann(), 4))   # 0.0

g3 = ghz(3)
print("GHZ(3) per-qubit entropies:",
      [round(g3.as_density_matrix().partial_trace({q}).entropy_von_neumann(), 4) for q in range(3)])

## Recap and what's next

- A qubit state is a normalized complex vector of **amplitudes**; $n$ qubits need $2^n$ of them. That **memory wall** is Feynman's founding argument for building quantum computers. The catch is that measurement returns only $n$ classical bits, so algorithms must arrange **interference** to make those few bits worth reading.
- Four states recur throughout the tutorial: $|0\rangle, |1\rangle$ (the poles, the **$Z$ basis**) and $|+\rangle, |-\rangle$ (the equator, the **$X$ basis**). $|+\rangle$ and $|-\rangle$ share the same measurement probabilities but differ by a **relative phase**, visible on the Bloch sphere.
- **Measurement** follows the **Born rule** ($P = |\text{amplitude}|^2$) and **collapses** the state. Deliberately chosen amplitudes give physics-guaranteed randomness: a shipping product (QRNG), and the loaded-coin version you built yourself.
- The **expectation value** $\langle O\rangle = \langle\psi|O|\psi\rangle$ (bra @ operator @ ket) is the single number that annealing (Part 3) and variational algorithms (Part 4) minimize. It reads a value off a state without changing the state, which is why it belongs here among the state primitives.
- **Entanglement** is correlated randomness shared only by its holders: perfectly correlated bits for insiders ($\ln 2$ nats of entropy per Bell qubit), pure noise ($\tfrac12 I$) for anyone outside.

**Part 2, Circuits / Digital Quantum Computing:** states are now *operated on* by **gates**, assembled into **circuits**, and **run** on the `QiliSim` simulator. That is where interference becomes visible, where measuring in a rotated basis lets you detect an eavesdropper (the BB84 protocol), and where we build a provably fair quantum dice roller.